# Actuarial Add-In Benchmark Tests

This notebook compares the Actuarial Add-In functions against established Python libraries:
- **scipy.stats** for probability distributions
- **chainladder** for reserving methods
- **numpy** for general numerical operations

Run this notebook to validate the accuracy of the add-in functions.

In [ ]:
# Install required packages if not already installed
# !pip install scipy numpy pandas chainladder matplotlib

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import chainladder as cl
import warnings
warnings.filterwarnings('ignore')

print(f"chainladder version: {cl.__version__}")
print(f"scipy version: {stats.__name__}")

## 1. Statistical Distributions

Compare distribution functions against scipy.stats.

### 1.1 Poisson Distribution

In [ ]:
# Poisson Distribution (lambda=5)
# Compare against: ACT_POISSON_PDF, ACT_POISSON_CDF, ACT_POISSON_INV

lambda_param = 5
k_values = list(range(11))

poisson_results = pd.DataFrame({
    'k': k_values,
    'scipy_pmf': [stats.poisson.pmf(k, lambda_param) for k in k_values],
    'scipy_cdf': [stats.poisson.cdf(k, lambda_param) for k in k_values],
})

# Expected values from add-in (from test_results.md)
addin_pmf = [0.006738, 0.033690, 0.084224, 0.140374, 0.175467, 
             0.175467, 0.146223, 0.104445, 0.065278, 0.036266, 0.018133]
addin_cdf = [0.006738, 0.040428, 0.124652, 0.265026, 0.440493,
             0.615961, 0.762183, 0.866628, 0.931906, 0.968172, 0.986305]

poisson_results['addin_pmf'] = addin_pmf
poisson_results['addin_cdf'] = addin_cdf
poisson_results['pmf_diff'] = abs(poisson_results['scipy_pmf'] - poisson_results['addin_pmf'])
poisson_results['cdf_diff'] = abs(poisson_results['scipy_cdf'] - poisson_results['addin_cdf'])

print("Poisson Distribution (λ=5)")
print("="*60)
print(poisson_results.round(6).to_string(index=False))
print(f"\nMax PMF difference: {poisson_results['pmf_diff'].max():.2e}")
print(f"Max CDF difference: {poisson_results['cdf_diff'].max():.2e}")

# Inverse CDF test
scipy_inv = stats.poisson.ppf(0.5, lambda_param)
addin_inv = 5  # From test_results.md
print(f"\nInverse CDF at p=0.5: scipy={scipy_inv}, add-in={addin_inv}")

### 1.2 Negative Binomial Distribution

In [ ]:
# Negative Binomial Distribution (r=5, p=0.3)
# Compare against: ACT_NEGBIN_PDF, ACT_NEGBIN_CDF, ACT_NEGBIN_INV

r, p = 5, 0.3
k_values = list(range(11))

# Note: scipy uses n=r (number of successes) and p=probability of success
negbin_results = pd.DataFrame({
    'k': k_values,
    'scipy_pmf': [stats.nbinom.pmf(k, r, p) for k in k_values],
    'scipy_cdf': [stats.nbinom.cdf(k, r, p) for k in k_values],
})

# Expected values from add-in
addin_pmf = [0.002430, 0.008505, 0.017860, 0.029172, 0.040841,
             0.051460, 0.060036, 0.066040, 0.069342, 0.070112, 0.068710]
addin_cdf = [0.002430, 0.010935, 0.028795, 0.057968, 0.098809,
             0.150268, 0.210305, 0.276345, 0.345686, 0.415799, 0.484509]

negbin_results['addin_pmf'] = addin_pmf
negbin_results['addin_cdf'] = addin_cdf
negbin_results['pmf_diff'] = abs(negbin_results['scipy_pmf'] - negbin_results['addin_pmf'])
negbin_results['cdf_diff'] = abs(negbin_results['scipy_cdf'] - negbin_results['addin_cdf'])

print("Negative Binomial Distribution (r=5, p=0.3)")
print("="*60)
print(negbin_results.round(6).to_string(index=False))
print(f"\nMax PMF difference: {negbin_results['pmf_diff'].max():.2e}")
print(f"Max CDF difference: {negbin_results['cdf_diff'].max():.2e}")

### 1.3 Lognormal Distribution

In [ ]:
# Lognormal Distribution (μ=0, σ=1)
# Compare against: ACT_LOGNORM_PDF, ACT_LOGNORM_CDF, ACT_LOGNORM_INV

mu, sigma = 0, 1
x_values = [0.5, 1, 1.5, 2, 3, 5]

lognorm_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.lognorm.pdf(x, sigma, scale=np.exp(mu)) for x in x_values],
    'scipy_cdf': [stats.lognorm.cdf(x, sigma, scale=np.exp(mu)) for x in x_values],
})

# Expected values from add-in
addin_pdf = [0.627496, 0.398942, 0.244974, 0.156874, 0.072728, 0.021851]
addin_cdf = [0.244109, 0.500000, 0.657432, 0.755891, 0.864031, 0.946240]

lognorm_results['addin_pdf'] = addin_pdf
lognorm_results['addin_cdf'] = addin_cdf
lognorm_results['pdf_diff'] = abs(lognorm_results['scipy_pdf'] - lognorm_results['addin_pdf'])
lognorm_results['cdf_diff'] = abs(lognorm_results['scipy_cdf'] - lognorm_results['addin_cdf'])

print("Lognormal Distribution (μ=0, σ=1)")
print("="*60)
print(lognorm_results.round(6).to_string(index=False))
print(f"\nMax PDF difference: {lognorm_results['pdf_diff'].max():.2e}")
print(f"Max CDF difference: {lognorm_results['cdf_diff'].max():.2e}")

# Inverse CDF test
scipy_inv = stats.lognorm.ppf(0.5, sigma, scale=np.exp(mu))
addin_inv = 1.0
print(f"\nInverse CDF at p=0.5: scipy={scipy_inv:.6f}, add-in={addin_inv:.6f}")

### 1.4 Gamma Distribution

In [ ]:
# Gamma Distribution (α=2, β=1)
# Compare against: ACT_GAMMA_PDF, ACT_GAMMA_CDF, ACT_GAMMA_INV

alpha, beta = 2, 1
x_values = [0.5, 1, 2, 3, 5]

# scipy uses shape=alpha, scale=1/beta
gamma_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.gamma.pdf(x, alpha, scale=1/beta) for x in x_values],
    'scipy_cdf': [stats.gamma.cdf(x, alpha, scale=1/beta) for x in x_values],
})

# Expected values from add-in
addin_pdf = [0.303265, 0.367879, 0.270671, 0.149361, 0.033690]
addin_cdf = [0.090204, 0.264241, 0.593994, 0.800852, 0.959572]

gamma_results['addin_pdf'] = addin_pdf
gamma_results['addin_cdf'] = addin_cdf
gamma_results['pdf_diff'] = abs(gamma_results['scipy_pdf'] - gamma_results['addin_pdf'])
gamma_results['cdf_diff'] = abs(gamma_results['scipy_cdf'] - gamma_results['addin_cdf'])

print("Gamma Distribution (α=2, β=1)")
print("="*60)
print(gamma_results.round(6).to_string(index=False))
print(f"\nMax PDF difference: {gamma_results['pdf_diff'].max():.2e}")
print(f"Max CDF difference: {gamma_results['cdf_diff'].max():.2e}")

# Inverse CDF test
scipy_inv = stats.gamma.ppf(0.5, alpha, scale=1/beta)
addin_inv = 1.678347
print(f"\nInverse CDF at p=0.5: scipy={scipy_inv:.6f}, add-in={addin_inv}")

### 1.5 Pareto Distribution

In [ ]:
# Pareto Distribution (α=2, xm=1)
# Compare against: ACT_PARETO_PDF, ACT_PARETO_CDF, ACT_PARETO_INV

alpha, xm = 2, 1
x_values = [1, 1.5, 2, 3, 5, 10]

# scipy pareto: scale=xm, shape=alpha (but PDF formula is different, need to adjust)
# scipy pareto uses: f(x) = alpha / x^(alpha+1) for x >= 1 with scale=xm
pareto_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.pareto.pdf(x/xm, alpha) / xm for x in x_values],
    'scipy_cdf': [stats.pareto.cdf(x/xm, alpha) for x in x_values],
})

# Expected values from add-in
addin_pdf = [2.000000, 0.592593, 0.250000, 0.074074, 0.016000, 0.002000]
addin_cdf = [0.000000, 0.555556, 0.750000, 0.888889, 0.960000, 0.990000]

pareto_results['addin_pdf'] = addin_pdf
pareto_results['addin_cdf'] = addin_cdf
pareto_results['pdf_diff'] = abs(pareto_results['scipy_pdf'] - pareto_results['addin_pdf'])
pareto_results['cdf_diff'] = abs(pareto_results['scipy_cdf'] - pareto_results['addin_cdf'])

print("Pareto Distribution (α=2, xm=1)")
print("="*60)
print(pareto_results.round(6).to_string(index=False))
print(f"\nMax PDF difference: {pareto_results['pdf_diff'].max():.2e}")
print(f"Max CDF difference: {pareto_results['cdf_diff'].max():.2e}")

# Inverse CDF test
scipy_inv = stats.pareto.ppf(0.5, alpha) * xm
addin_inv = 1.414214
print(f"\nInverse CDF at p=0.5: scipy={scipy_inv:.6f}, add-in={addin_inv}")

### 1.6 Weibull Distribution

In [ ]:
# Weibull Distribution (k=2, λ=1)
# Compare against: ACT_WEIBULL_PDF, ACT_WEIBULL_CDF, ACT_WEIBULL_INV

k, lam = 2, 1
x_values = [0.5, 1, 1.5, 2, 3]

# scipy.stats.weibull_min: c=k (shape), scale=λ
weibull_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.weibull_min.pdf(x, k, scale=lam) for x in x_values],
    'scipy_cdf': [stats.weibull_min.cdf(x, k, scale=lam) for x in x_values],
})

print("Weibull Distribution (k=2, λ=1) - Benchmark Values")
print("="*60)
print(weibull_results.round(6).to_string(index=False))

# Inverse CDF
for p in [0.25, 0.5, 0.75]:
    inv = stats.weibull_min.ppf(p, k, scale=lam)
    print(f"Inverse CDF at p={p}: {inv:.6f}")

### 1.7 Beta Distribution

In [ ]:
# Beta Distribution (α=2, β=5)
# Compare against: ACT_BETA_PDF, ACT_BETA_CDF, ACT_BETA_INV

a, b = 2, 5
x_values = [0.1, 0.2, 0.3, 0.5, 0.7]

beta_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.beta.pdf(x, a, b) for x in x_values],
    'scipy_cdf': [stats.beta.cdf(x, a, b) for x in x_values],
})

print("Beta Distribution (α=2, β=5) - Benchmark Values")
print("="*60)
print(beta_results.round(6).to_string(index=False))

# Inverse CDF
for p in [0.25, 0.5, 0.75]:
    inv = stats.beta.ppf(p, a, b)
    print(f"Inverse CDF at p={p}: {inv:.6f}")

### 1.8 Generalized Pareto Distribution (GPD)

In [ ]:
# Generalized Pareto Distribution (ξ=0.5, σ=1, μ=0)
# Compare against: ACT_GPD_PDF, ACT_GPD_CDF, ACT_GPD_INV

xi, sigma, mu = 0.5, 1, 0
x_values = [0.5, 1, 2, 3, 5]

# scipy.stats.genpareto: c=xi (shape), scale=sigma, loc=mu
gpd_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.genpareto.pdf(x, xi, loc=mu, scale=sigma) for x in x_values],
    'scipy_cdf': [stats.genpareto.cdf(x, xi, loc=mu, scale=sigma) for x in x_values],
})

print("Generalized Pareto Distribution (ξ=0.5, σ=1, μ=0) - Benchmark Values")
print("="*60)
print(gpd_results.round(6).to_string(index=False))

# Inverse CDF
for p in [0.25, 0.5, 0.75, 0.9, 0.99]:
    inv = stats.genpareto.ppf(p, xi, loc=mu, scale=sigma)
    print(f"Inverse CDF at p={p}: {inv:.6f}")

### 1.9 Burr Type XII Distribution

In [ ]:
# Burr Type XII Distribution (c=2, k=3, λ=1)
# Compare against: ACT_BURR_PDF, ACT_BURR_CDF, ACT_BURR_INV

c, k_param, lam = 2, 3, 1
x_values = [0.5, 1, 2, 3, 5]

# scipy.stats.burr12: c and d parameters (d=k in actuarial notation)
burr_results = pd.DataFrame({
    'x': x_values,
    'scipy_pdf': [stats.burr12.pdf(x, c, k_param, scale=lam) for x in x_values],
    'scipy_cdf': [stats.burr12.cdf(x, c, k_param, scale=lam) for x in x_values],
})

print("Burr Type XII Distribution (c=2, k=3, λ=1) - Benchmark Values")
print("="*60)
print(burr_results.round(6).to_string(index=False))

# Inverse CDF
for p in [0.25, 0.5, 0.75, 0.9]:
    inv = stats.burr12.ppf(p, c, k_param, scale=lam)
    print(f"Inverse CDF at p={p}: {inv:.6f}")

## 2. Chain Ladder Methods

Compare reserving methods against the chainladder Python package.

In [ ]:
# Create sample triangle matching test_results.md
triangle_data = np.array([
    [100, 150, 170, 180, 185],
    [110, 165, 190, 200, np.nan],
    [120, 180, 210, np.nan, np.nan],
    [130, 195, np.nan, np.nan, np.nan],
    [140, np.nan, np.nan, np.nan, np.nan]
])

# Create chainladder Triangle
triangle = cl.Triangle(
    pd.DataFrame(triangle_data, 
                 index=[2019, 2020, 2021, 2022, 2023],
                 columns=[1, 2, 3, 4, 5]),
    origin='index',
    development='columns',
    cumulative=True
)

print("Input Triangle:")
print(triangle)

### 2.1 Development Factors

In [ ]:
# Calculate development factors using chainladder
dev = cl.Development().fit(triangle)

print("Development Factors (chainladder):")
print(dev.ldf_)

# Expected from add-in
addin_factors = [1.5000, 1.1515, 1.0556, 1.0278]
print("\nExpected from add-in:")
print(addin_factors)

### 2.2 Chain Ladder Ultimate

In [ ]:
# Chain Ladder Ultimate estimates
cl_model = cl.Chainladder().fit(dev.transform(triangle))

print("Chain Ladder Ultimate (chainladder):")
print(cl_model.ultimate_)

# Expected from add-in
addin_ultimates = [185.00, 205.56, 227.82, 243.60, 262.34]
print("\nExpected from add-in:")
for i, (ay, ult) in enumerate(zip([2019, 2020, 2021, 2022, 2023], addin_ultimates)):
    print(f"  AY {ay}: {ult:.2f}")

### 2.3 IBNR Reserves

In [ ]:
# IBNR calculation
print("IBNR Reserves (chainladder):")
print(cl_model.ibnr_)

# Expected from add-in
addin_ibnr = [0.00, 5.56, 17.82, 48.60, 122.34]
print("\nExpected from add-in:")
for i, (ay, ibnr) in enumerate(zip([2019, 2020, 2021, 2022, 2023], addin_ibnr)):
    print(f"  AY {ay}: {ibnr:.2f}")
print(f"  Total: {sum(addin_ibnr):.2f}")

### 2.4 Mack Standard Errors

In [ ]:
# Mack Chain Ladder with standard errors
mack = cl.MackChainladder().fit(dev.transform(triangle))

print("Mack Standard Errors (chainladder):")
print(mack.full_std_err_)

# Expected from add-in
addin_se = [0.00, 0.00, 1.16, 4.56, 4.78]
print("\nExpected from add-in:")
for i, (ay, se) in enumerate(zip([2019, 2020, 2021, 2022, 2023], addin_se)):
    print(f"  AY {ay}: {se:.2f}")

### 2.5 Bornhuetter-Ferguson

In [ ]:
# Bornhuetter-Ferguson method
# Requires a-priori ultimate estimates
apriori = 250  # Example a-priori ultimate

bf = cl.BornhuetterFerguson(apriori=apriori).fit(
    dev.transform(triangle)
)

print(f"Bornhuetter-Ferguson (a-priori={apriori}):")
print("Ultimate:")
print(bf.ultimate_)
print("\nIBNR:")
print(bf.ibnr_)

### 2.6 Cape Cod Method

In [ ]:
# Cape Cod method
# Requires premium or exposure data
premium = pd.Series([200, 220, 240, 260, 280], index=[2019, 2020, 2021, 2022, 2023])

cape_cod = cl.CapeCod().fit(
    dev.transform(triangle),
    sample_weight=premium
)

print("Cape Cod Method:")
print("Ultimate:")
print(cape_cod.ultimate_)
print("\nIBNR:")
print(cape_cod.ibnr_)
print(f"\nEstimated ELR: {cape_cod.apriori_:.4f}")

### 2.7 Bootstrap Chain Ladder

In [ ]:
# Bootstrap Chain Ladder for reserve distribution
boot = cl.BootstrapODPSample(n_sims=1000, random_state=42).fit(
    dev.transform(triangle)
)

# Get percentiles
print("Bootstrap Reserve Distribution:")
print(f"Mean IBNR: {boot.ibnr_.mean():.2f}")
print(f"Std Dev: {boot.ibnr_.std():.2f}")

# Percentiles
for pct in [0.50, 0.75, 0.90, 0.95, 0.99]:
    val = np.percentile(boot.ibnr_.values.flatten(), pct*100)
    print(f"{pct:.0%} percentile: {val:.2f}")

## 3. Taylor-Ashe Sample Data

Test against the famous Taylor-Ashe dataset used in actuarial literature.

In [ ]:
# Load Taylor-Ashe sample data from chainladder
ta = cl.load_sample('taylor_ashe')

print("Taylor-Ashe Triangle:")
print(ta)

# Fit chain ladder
ta_dev = cl.Development().fit(ta)
ta_cl = cl.Chainladder().fit(ta_dev.transform(ta))

print("\nDevelopment Factors:")
print(ta_dev.ldf_)

print("\nUltimate Estimates:")
print(ta_cl.ultimate_)

print("\nIBNR:")
print(ta_cl.ibnr_)
print(f"\nTotal IBNR: {float(ta_cl.ibnr_.sum()):.0f}")

In [ ]:
# Mack analysis on Taylor-Ashe
ta_mack = cl.MackChainladder().fit(ta_dev.transform(ta))

print("Mack Chain Ladder on Taylor-Ashe:")
print("\nProcess Standard Error:")
print(ta_mack.process_risk_)
print("\nParameter Standard Error:")
print(ta_mack.parameter_risk_)
print("\nTotal Standard Error:")
print(ta_mack.total_mack_std_err_)

## 4. Exposure Curves

Benchmark exposure curve calculations.

In [ ]:
def mbbefd_curve(d, b, g):
    """MBBEFD exposure curve."""
    if d <= 0:
        return 0
    if d >= 1:
        return 1
    if b == 1 and g == 1:
        return d
    if b == 1:
        return (np.log(1 + (g - 1) * d)) / np.log(g)
    if g == 1:
        return (1 - b**d) / (1 - b)
    return np.log((g - 1 + b**d) / g) / np.log(b)

# Test MBBEFD curve (b=2, g=3)
b, g = 2, 3
d_values = np.arange(0, 1.1, 0.1)

mbbefd_results = pd.DataFrame({
    'd': d_values,
    'G(d)': [mbbefd_curve(d, b, g) for d in d_values]
})

# Expected from add-in
addin_mbbefd = [0.000000, 0.149001, 0.278578, 0.394114, 0.499046,
                0.595704, 0.685740, 0.770368, 0.850505, 0.926862, 1.000000]
mbbefd_results['addin_G(d)'] = addin_mbbefd
mbbefd_results['diff'] = abs(mbbefd_results['G(d)'] - mbbefd_results['addin_G(d)'])

print("MBBEFD Curve (b=2, g=3)")
print("="*50)
print(mbbefd_results.round(6).to_string(index=False))
print(f"\nMax difference: {mbbefd_results['diff'].max():.2e}")

In [ ]:
def swissre_curve(d, curve_type):
    """Swiss Re exposure curves (1-5)."""
    # Swiss Re curve parameters from Bernegger (1997)
    params = {
        1: (1.5, 1.5),   # Light
        2: (2.0, 2.0),   # Light-Medium
        3: (3.0, 3.0),   # Medium
        4: (4.0, 4.0),   # Medium-Heavy
        5: (5.0, 5.0),   # Heavy
    }
    b, g = params[curve_type]
    return mbbefd_curve(d, b, g)

# Test Swiss Re curves at d=0.5
d = 0.5
swissre_results = pd.DataFrame({
    'Curve': [1, 2, 3, 4, 5],
    'Python G(0.5)': [swissre_curve(d, c) for c in range(1, 6)],
    'Add-in G(0.5)': [0.525127, 0.582599, 0.713919, 0.803283, 0.854155]
})
swissre_results['diff'] = abs(swissre_results['Python G(0.5)'] - swissre_results['Add-in G(0.5)'])

print("Swiss Re Curves at d=0.5")
print("="*50)
print(swissre_results.round(6).to_string(index=False))

## 5. Credibility Theory

Benchmark credibility calculations.

In [ ]:
def buhlmann_credibility(n, k):
    """Bühlmann credibility factor Z = n / (n + k)."""
    return n / (n + k)

def buhlmann_straub_credibility(weights, k):
    """Bühlmann-Straub credibility factor Z = sum(w) / (sum(w) + k)."""
    total_weight = sum(weights)
    return total_weight / (total_weight + k)

# Test Bühlmann credibility
test_cases = [
    (5, 10),
    (10, 10),
    (20, 10),
    (50, 10),
    (100, 10),
]

print("Bühlmann Credibility (k=10)")
print("="*40)
for n, k in test_cases:
    z = buhlmann_credibility(n, k)
    print(f"n={n:3d}: Z = {z:.4f}")

print("\nBühlmann-Straub Credibility")
print("="*40)
weights = [100, 150, 200, 250, 300]
k = 500
z = buhlmann_straub_credibility(weights, k)
print(f"Weights: {weights}")
print(f"k: {k}")
print(f"Z = {z:.4f}")

## 6. Summary Report

Generate a summary of all benchmark comparisons.

In [ ]:
print("="*70)
print("ACTUARIAL ADD-IN BENCHMARK SUMMARY")
print("="*70)
print()
print("Distribution Functions:")
print(f"  - Poisson:          Max diff = {poisson_results['pmf_diff'].max():.2e}")
print(f"  - Negative Binomial: Max diff = {negbin_results['pmf_diff'].max():.2e}")
print(f"  - Lognormal:        Max diff = {lognorm_results['pdf_diff'].max():.2e}")
print(f"  - Gamma:            Max diff = {gamma_results['pdf_diff'].max():.2e}")
print(f"  - Pareto:           Max diff = {pareto_results['pdf_diff'].max():.2e}")
print()
print("Exposure Curves:")
print(f"  - MBBEFD:           Max diff = {mbbefd_results['diff'].max():.2e}")
print()
print("Chain Ladder Methods:")
print("  - See detailed comparison above")
print()
print("="*70)
print("All benchmarks completed successfully!")
print("="*70)